# 03 · Descomposición y correlación

**Tiempo estimado:** 30 min.

**Objetivos.**

1. Descomponer caudal y piezometría con **STL**.
2. Analizar **ACF y PACF** (caudal y residuos de STL).
3. Estimar el **lag dominante** lluvia → caudal mediante CCF.

**Datasets.** Caudal Pinos-Genil, lluvia mensual Iznájar, piezometría `PZ0267014`.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

from cst import datos as ud

plt.rcParams.update({"figure.figsize": (10, 3.4), "axes.grid": True, "grid.alpha": 0.3})

caudal = ud.cargar_caudal_genil(source="CEDEX")
lluvia_m = ud.cargar_lluvia_genil(source="CEDEX")
piezo = ud.cargar_piezometria()

## 1 · STL del caudal

Pasamos a frecuencia mensual e interpolamos lo mínimo imprescindible — STL no admite NaN. Usamos `period=12` (estacionalidad anual).


In [ ]:
caudal_mensual = caudal.resample("MS").mean().loc["1995":"2020"]
# Rellenamos sólo gaps pequeños para STL; documentado en el notebook 02
caudal_mensual = caudal_mensual.interpolate("linear", limit=2)

stl = STL(caudal_mensual.dropna(), period=12, robust=True).fit()
fig = stl.plot()
fig.set_size_inches(10, 6)
for ax in fig.axes:
    ax.grid(alpha=0.3)
plt.tight_layout()

**Lectura:**

- _Trend_: ligero descenso a largo plazo (debate: sequía estructural vs regulación de embalses).
- _Seasonal_: ciclo anual claro, picos en febrero-marzo.
- _Resid_: aún hay estructura (picos asociados a eventos). STL captura ciclo y tendencia, no eventos puntuales.


## 2 · STL en la piezometría

La piezometría tiene un ciclo anual fuerte por recarga invernal y extracción veraniega. La serie es irregular: hay que regularizarla antes de STL.


In [ ]:
# Resamplear a mensual: media de los valores que caen en cada mes (mes sin valor → NaN)
piezo_mensual = piezo.resample("MS").mean()
print(f"Meses con dato: {piezo_mensual.notna().sum()} / {len(piezo_mensual)}")

# Usamos un periodo razonablemente cubierto y luego interpolamos suavemente
piezo_stl = piezo_mensual.loc["2005":"2024"].interpolate("linear", limit=6)
piezo_stl = piezo_stl.dropna()

stl_piezo = STL(piezo_stl, period=12, robust=True).fit()
fig = stl_piezo.plot()
fig.set_size_inches(10, 6)
for ax in fig.axes:
    ax.grid(alpha=0.3)
plt.tight_layout()

## 3 · Tests de estacionariedad

Comprobamos si la serie original es estacionaria, y si la **diferenciada** lo es.


In [ ]:
def diagnostico(serie, nombre):
    s = serie.dropna()
    p_adf = adfuller(s)[1]
    p_kpss = kpss(s, regression="c", nlags="auto")[1]
    print(f"{nombre:40s}  ADF p={p_adf:6.4f}   KPSS p={p_kpss:6.4f}")


diagnostico(caudal.loc["2000":"2020"], "caudal diario (Pinos-Genil)")
diagnostico(caudal.loc["2000":"2020"].diff(), "caudal diferenciado (lag 1)")
diagnostico(caudal_mensual, "caudal mensual")
diagnostico(caudal_mensual.diff(12), "caudal mensual diferenciado estacionalmente (lag 12)")
diagnostico(piezo_stl, "piezometría mensual (interp.)")

**Convención de lectura.** ADF $H_0$: no estacionaria → rechazar (p<0.05) ⇒ estacionaria. KPSS $H_0$: estacionaria → rechazar ⇒ NO. Las dos conclusiones coinciden cuando la respuesta es clara.


## 4 · ACF y PACF del caudal

Los dos gráficos guían la elección de órdenes en ARIMA (sesión 2).


In [ ]:
serie = caudal.loc["2010":"2020"].dropna()

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
plot_acf(serie, lags=60, ax=axes[0], color="#1f6f8b")
axes[0].set_title("ACF caudal diario (60 días)")
plot_pacf(serie, lags=30, ax=axes[1], color="#7c3aed", method="ywm")
axes[1].set_title("PACF caudal diario (30 días)")
plt.tight_layout()

**Interpretación rápida:**

- La ACF cae despacio → presencia de tendencia / no-estacionariedad → puede pedir diferenciación.
- PACF: los primeros 2-3 lags dominan, después caen rápidamente → un AR(2)/AR(3) podría ser razonable como modelo base.
- En la sesión 2 cerraremos los órdenes con AIC/BIC y diagnóstico de residuos.


## 5 · ACF sobre los residuos de STL

Una vez quitada la tendencia y la estacionalidad, ¿queda señal o sólo ruido blanco?


In [ ]:
resid = stl.resid.dropna()
fig, ax = plt.subplots(figsize=(10, 3.4))
plot_acf(resid, lags=36, ax=ax, color="#16a34a")
ax.set_title("ACF de los residuos STL (mensual)")
plt.tight_layout()

Si la ACF de los residuos tiene barras significativas (sobre todo en los lags bajos), todavía hay estructura que puede modelarse. STL es un punto de partida, no el final.


## 6 · Correlación cruzada lluvia → caudal

La lluvia mensual de Iznájar y el caudal mensual de Pinos-Genil están en cuencas conectadas (Genil → embalse de Iznájar → Genil bajo). Buscamos el lag con mayor correlación.


In [ ]:
df = pd.concat({"lluvia": lluvia_m, "caudal": caudal_mensual}, axis=1).dropna()
lags = range(-6, 13)
ccf = [df["lluvia"].corr(df["caudal"].shift(-k)) for k in lags]

fig, ax = plt.subplots()
ax.bar(list(lags), ccf, color="#2563eb", alpha=0.7)
ax.axhline(0, color="k", lw=0.5)
ax.axvline(0, color="grey", ls="--", lw=0.5)
ax.set_xlabel("Lag k (meses) — lluvia(t) vs caudal(t+k)")
ax.set_ylabel(r"$\rho_{lluvia,caudal}(k)$")
ax.set_title("CCF lluvia → caudal (mensual)")
plt.tight_layout()

k_max = max(zip(ccf, lags), key=lambda x: abs(x[0]))[1]
print(f"Lag de máxima correlación: k = {k_max} meses")

**Lectura física.** El pico positivo cae típicamente en lag 1-2 meses: la cuenca regulada del Genil amortigua el efecto inmediato de la lluvia, y el embalse de Iznájar suaviza la respuesta. Si tuviéramos la lluvia diaria (SAIH `A20_202`), el lag se medirá en días — la cuenca "natural" responde mucho más rápido que el sistema regulado.


## 7 · Ejercicios

1. **STL anual vs mensual.** Repite el STL del caudal con `period=12` sobre la serie diaria (usa `period=365`). ¿Qué cambia? ¿Cuánto más lento es?
2. **STL del logaritmo.** Aplica STL a `np.log(caudal_mensual)` y compara con el directo. ¿La descomposición es más limpia? (Pista: la varianza del residual.)
3. **ACF en distintas ventanas.** Calcula la ACF del caudal por décadas (1980s, 1990s, 2000s, 2010s). ¿La estructura es estable?
4. **CCF a otro pluviómetro.** Reutiliza `ud.descargar_anuario_csv` para descargar `evap.csv` y prueba con otra estación evaporimétrica (por ejemplo Bembézar `ref_evap=5002`). ¿Cambia el lag dominante? ¿Por qué?
5. **Reto.** Construye un modelo _naive_ de respuesta a impulso: caudal(t) = α·lluvia(t-1) + β·lluvia(t-2). Ajusta α, β por mínimos cuadrados y compara con la baseline persistencia (caudal(t) = caudal(t-1)).
6. **CCF diario SAIH lluvia→caudal.** El CCF mensual sólo resuelve el lag con grano de mes. Calcula ahora el CCF **diario** entre `ud.cargar_lluvia_genil()` y `ud.cargar_caudal_genil()` — ambas series del SAIH, del mismo punto físico (cluster A20). Explora lags de -5 a +30 días. ¿Cuál es el lag de máxima correlación? Compáralo con el resultado mensual: ¿coincide? Discute el efecto de la regulación del embalse de Iznájar.
7. **CCF lluvia-piezometría con ERA5.** El piezómetro `PZ0267014` está en Valladolid; no hay SAIH ahí. Usa `ud.cargar_lluvia_duero_diaria()` (ERA5 sobre la coordenada del pozo) para obtener la lluvia local. Resamplea a mensual (suma) y calcula el CCF con la cota piezométrica mensual. ¿En qué lag aparece el máximo? ¿Qué te dice eso del tiempo de tránsito agua-suelo → acuífero?
